# Ticket scarcity and appropriate reliance on AI
Qianshuo Wang | PS1 working draft

Synthetic classroom model. No human participants and no real AI model. Uses Python standard library only. Run all cells in Google Colab. This file was executed locally cell-by-cell; hosted Colab verification is pending.

The original game uses deterministic scenario hashes and retrospective correctness. Here we explicitly assume a Bernoulli sale-out state and a symmetric noisy state signal with conditional accuracy r. This stochastic extension is not an estimate of the live game’s calibration.

In [1]:
"""Synthetic ticket decision model. No human data or actual AI predictions."""
import math, random, csv, json
from pathlib import Path

def risk(Q,N):
    if not math.isfinite(Q) or not math.isfinite(N) or Q < 1 or N < 0:
        raise ValueError('Q must be positive; N must be nonnegative')
    return min(.98, N/(N+8*Q+1))

def posterior(p,r,advice):
    if not (0<=p<=1 and 0<=r<=1) or advice not in ('buy','wait'):
        raise ValueError('invalid probability or advice')
    # Symmetric noisy signal of sale-out state; not an empirical accuracy claim.
    a,b=(r,1-r) if advice=='buy' else (1-r,r)
    denom=p*a+(1-p)*b
    return None if denom==0 else p*a/denom

def values(P,V,p):
    if not (math.isfinite(P) and math.isfinite(V) and 0<P<V and 0<=p<=1):
        raise ValueError('analysis restricts to 0 < P < V and p in [0,1]')
    return V-P,(1-p)*(V-.9*P)

def action(P,V,p):
    b,w=values(P,V,p)
    return 'buy' if b>=w else 'wait'

def expected(P,V,p,r,policy):
    total=0.
    for sold in (True,False):
        for advice in ('buy','wait'):
            prob=(p if sold else 1-p)*(r if ((advice=='buy')==sold) else 1-r)
            if prob==0: continue
            post=posterior(p,r,advice)
            choice=advice if policy=='follow' else action(P,V,p if policy=='ignore' else post)
            reward=V-P if choice=='buy' else (0 if sold else V-.9*P)
            total+=prob*reward
    return total

def run(outdir):
    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    rows=[]
    for N in (10,50,100,500,900):
      for r in (.3,.5,.6,.8,1.):
        p=risk(40,N)
        row={'N':N,'Q':40,'P':150,'V':220,'reliability':r,'risk':p}
        for policy in ('ignore','follow','bayes'):
            row[policy]=expected(150,220,p,r,policy)
        row['follow_regret']=row['bayes']-row['follow']
        rows.append(row)
    with (outdir/'policy_comparison.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=rows[0]);writer.writeheader();writer.writerows(rows)
    p=risk(40,500);r=.8
    rng=random.Random(206); totals={k:0. for k in ('ignore','follow','bayes')}
    trials=100000
    for _ in range(trials):
        sold=rng.random()<p;correct=rng.random()<r
        advice='buy' if sold==correct else 'wait'
        for policy in totals:
            choice=advice if policy=='follow' else action(150,220,p if policy=='ignore' else posterior(p,r,advice))
            totals[policy]+=70 if choice=='buy' else (0 if sold else 85)
    result={'evidence':'synthetic exact enumeration and Monte Carlo; not human behavior',
      'seed':206,'trials':trials,'risk':p,'no_advice_threshold':15/85,
      'posterior_buy':posterior(p,r,'buy'),'posterior_wait':posterior(p,r,'wait'),
      'exact':{k:expected(150,220,p,r,k) for k in totals},
      'monte_carlo':{k:v/trials for k,v in totals.items()},
      'grid_cases':len(rows)}
    (outdir/'summary.json').write_text(json.dumps(result,indent=2)+'\n')
    return result



## Exact enumeration and one modification
Baseline: follow every recommendation. Modification: condition decisions on the Bayesian posterior using the same state-signal distribution. Include an ignore-advice comparator. Each condition enumerates two states and two recommendations. Compare expected payoff and regret to the Bayesian optimum.

No-advice threshold: buy if p >= 0.1P/(V-0.9P). Conditional on advice, replace p by its posterior. The main analysis restricts V>P>0; it does not silently clip negative surplus.

In [2]:
result=run('ps1_outputs')
print(json.dumps(result,indent=2))

{
  "evidence": "synthetic exact enumeration and Monte Carlo; not human behavior",
  "seed": 206,
  "trials": 100000,
  "risk": 0.6090133982947625,
  "no_advice_threshold": 0.17647058823529413,
  "posterior_buy": 0.8616975441619992,
  "posterior_wait": 0.2802690582959641,
  "exact": {
    "ignore": 70.0,
    "follow": 66.16565164433618,
    "bayes": 70.0
  },
  "monte_carlo": {
    "ignore": 70.0,
    "follow": 66.202,
    "bayes": 70.0
  },
  "grid_cases": 25
}


## Independent checks
Uninformative advice preserves prior beliefs; perfect and inverted signals recover states; posterior-optimal choice weakly dominates fixed comparison policies. Monte Carlo supplies an independent numerical check, not behavioral evidence.

In [3]:
assert abs(values(150,220,15/85)[0]-values(150,220,15/85)[1])<1e-10
for p in (0,.01,.1,.61,.98,1):
 for r in (0,.3,.5,.8,1):
  for policy in ('follow','ignore'):
   assert expected(150,220,p,r,'bayes')+1e-10>=expected(150,220,p,r,policy)
assert abs(result['monte_carlo']['follow']-result['exact']['follow'])<0.5
print('Threshold, posterior-policy dominance and Monte Carlo checks passed.')

Threshold, posterior-policy dominance and Monte Carlo checks passed.


## Interpretation and next human test
At default market inputs, even Wait advice at r=0.8 leaves posterior sale-out probability above the buy threshold. Therefore always following advice loses expected utility in this condition. This does not show that people overrely on AI. Proposed human validation separates reported sale-out beliefs from advice-following; outcomes and sample size remain planned.

Sources: Li, Lu & Yin (2023), https://doi.org/10.1609/aaai.v37i5.25748 ; Simon (1955), https://doi.org/10.2307/1884852. Neither paper supplies the numerical risk formula or signal calibration; those are classroom assumptions.

AI disclosure: Codex assisted in model formalization, coding and verification on September 12, 2026, after the author supplied Intellectual Statement II. Author review and independent rerun are pending.